# PyTorch for Education Rosters — Practice Skeleton

**Short name (GitHub):** `PTEdu`

**Lab source:** Education-sector adaptation of `PTIntro` / Codefinity *What is PyTorch* (tensors, factories, `nn.Module`, training loop). The table is a 180-row student roster instead of Iris.

Work this notebook first. Peek at `PTEdu_Solution.ipynb` only when you are stuck. Helpers live in `PTEdu.py`.

**Files you will use**
- `data/students_outcomes.csv` — 180 students × 4 numeric features + `outcome`
- `ptedu_flowchart.png` — desired outcome
- `PTEdu_Cheatsheet.docx` — one-page lookup
- `PTEdu.py` — split / scale / `OutcomeNet` / `train_roster`

**Not a grading engine.** Bands are teaching labels on a synthetic roster. Do not use this net to assign course grades, placements, or early-alert flags without a human review process that this lab does not provide.


## Inline cheat-sheet (keep this cell visible)

See also **`PTEdu_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Roster row | one student; columns = attendance %, study h/week, prior GPA, assignment avg |
| Bands | `support=0`, `on_track=1`, `honors=2` |
| Tensor ranks | scalar 0-D, vector 1-D, matrix 2-D |
| From a list | `torch.tensor([[1,2],[3,4]])` |
| Factories | `zeros`, `ones`, `arange` (end exclusive), `linspace` (both ends in), `*_like` |
| dtypes | features `float32`; class indices `long` |
| Scale | train mean/sd only — units are `%`, hours, 4.0 GPA, 100-pt avg |
| Module | `OutcomeNet`: `4 → hidden ReLU → 3` logits |
| Call | `model(x)` — **not** `model.forward(x)` |
| Loss | `nn.CrossEntropyLoss()` — do **not** softmax first |
| Train step | `zero_grad()` → forward → loss → `backward()` → `step()` |
| Eval | `model.eval()` **and** `with torch.no_grad():` |
| Labels | `torch.argmax(logits, dim=1)` |

**Education-specific gotcha:** raw units underfit (≈69% here). Scaled features reach ≈97%. Iris hid this because every column was centimetres.


## Desired outcome

![flowchart](ptedu_flowchart.png)

1. Create tensors from lists and factory helpers (`zeros`, `ones`, `arange`, `linspace`, `*_like`).
2. Load the roster, encode `outcome` as `0/1/2`, split 80/20 (12 students per band on test).
3. **Standardize with the training mean and sd** — then wrap as `float32` / `long`.
4. Define `OutcomeNet`: `4 → 16 ReLU → 3` logits.
5. Train with Adam + cross-entropy, 150 full-batch epochs.
6. `model.eval()` + `no_grad` + `argmax` → test accuracy.
7. Replay Sequential / SGD / class-mean baseline, then turn the simulation knobs.


## Setup


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)
np.random.seed(42)

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("device stays on CPU for this lab")
print("disclaimer: not a grading engine")


## 0. What is PyTorch, in an education shop?

PyTorch is a Python library for **tensors + autograd + neural-net modules**. The API does not change when the rows stop being flowers and start being students.

- **Dynamic graph** — `if`/`for` in `forward` are ordinary Python.
- **GPU path** — `.to("cuda")` when you have one; this lab stays on CPU.
- **`torch.nn`** — layers, losses, `Module`.
- **`autograd`** — `loss.backward()` fills `.grad` on every trained parameter.

Tensors are n-dimensional arrays (scalar / vector / matrix / batch). Indexing matches NumPy. The extras are device placement and the tape that records ops for backprop.

Why education data is a better first table than Iris for *scaling*: attendance is 40–100, study hours are 0–20, GPA is 0–4, assignment average is 35–100. A `Linear` layer treats those as the same kind of number. They are not. Section 4 forces a train-only z-score before the net sees the roster.


## 1. Create tensors from Python lists

**Task.** Create:

1. A 2-D tensor from `[[1, 2], [3, 4]]`.
2. A 3-D tensor **directly from a nested list literal** (no intermediate variable).

Print both tensors and their `.shape`.


In [ ]:
data = [[1, 2], [3, 4]]
tensor_2d = None  # TODO: torch.tensor(data)
tensor_3d = None  # TODO: torch.tensor([[[...], [...]], ...])

print(tensor_2d)
print("2d shape", None)
print(tensor_3d)
print("3d shape", None)


## 2. Factory helpers — zeros, ones, arange, linspace, like

**Task.**

1. `zeros` — a 2×3 float tensor (think “placeholder for 2 students × 3 features”).
2. `arange` — integers **1 through 10 inclusive**.
3. `linspace` — 10 evenly spaced values from 2 to 4 **inclusive** (a fake GPA grid).
4. `x = torch.tensor([[1, 2, 3], [4, 5, 6]])` then `zeros_like` / `ones_like`.


In [ ]:
zeros_23 = None
one_to_ten = None
gpa_grid = None

x = torch.tensor([[1, 2, 3], [4, 5, 6]])
zeros_like_x = None
ones_like_x = None

print(zeros_23)
print(one_to_ten)
print(gpa_grid)
print(zeros_like_x)
print(ones_like_x)


## 3. Load the roster and encode the target

180 students, 60 in each band: `support`, `on_track`, `honors`.

Four features, incompatible units on purpose:

- `attendance_pct` — 40 to 100
- `study_hours` — hours / week
- `prior_gpa` — 0.7 to 4.0
- `assignment_avg` — 35 to 100

**Task.** Read `data/students_outcomes.csv`. Build `X` (`float32`) and `y` (`support=0`, `on_track=1`, `honors=2`, `int64`). Print shape and class counts.


In [ ]:
roster = pd.read_csv("data/students_outcomes.csv")
print(roster.head(3))
print(roster["outcome"].value_counts())

feature_cols = ["attendance_pct", "study_hours", "prior_gpa", "assignment_avg"]
band_to_idx = {"support": 0, "on_track": 1, "honors": 2}

X = None  # TODO
y = None  # TODO

print("X", None, "y", None)
print("counts", None)


## 4. Stratified 80/20 split, scale, wrap as tensors

sklearn is **not** required. 60 per band → 12 test students each.

**Task.**

1. Split with the helper below (or `PTEdu.train_test_split`).
2. Standardize with the **training** mean and sd only.
3. Convert scaled `X_*` to `torch.float32` and `y_*` to `torch.long`.

Skip the scale step once on purpose if you want to see the 69% version. Keep the scaled tensors for the rest of the notebook.


In [ ]:
def train_test_split(X, y, test_size=0.2, random_state=42):
    rng = np.random.RandomState(random_state)
    tr, te = [], []
    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)
        n_te = int(round(len(idx) * test_size))
        te.append(idx[:n_te]); tr.append(idx[n_te:])
    tr = np.concatenate(tr); te = np.concatenate(te)
    rng.shuffle(tr); rng.shuffle(te)
    return X[tr], X[te], y[tr], y[te]

X_train, X_test, y_train, y_test = None  # TODO

mu = None   # TODO X_train.mean(0)
sd = None   # TODO X_train.std(0) + 1e-6
X_train_s = None
X_test_s = None

X_train_t = None
X_test_t = None
y_train_t = None
y_test_t = None

print("train", None, "test", None)


## 5. Define `OutcomeNet`

Same shape as the Iris net — the sector changed, the module contract did not.

- `__init__(self, input_size, hidden_size, output_size)`
- `super().__init__()`
- `self.fc1 = nn.Linear(input_size, hidden_size)`
- `self.fc2 = nn.Linear(hidden_size, output_size)`
- `forward`: ReLU after `fc1`, raw logits from `fc2`

Instantiate with `4`, `16`, `3`.


In [ ]:
class OutcomeNet(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        # TODO
        pass

    def forward(self, x):
        return x

model = None  # TODO OutcomeNet(4, 16, 3)
print(model)


## 6. Train — loss, optimizer, loop

- `criterion = nn.CrossEntropyLoss()`
- `optimizer = torch.optim.Adam(model.parameters(), lr=0.01)`
- 150 epochs, full batch (144 rows)

Each epoch: `zero_grad` → `model(X_train_t)` → loss → `backward` → `step`. Print every 15 epochs. Store `losses` for the plot.


In [ ]:
criterion = None
optimizer = None
epochs = 150
losses = []

for epoch in range(epochs):
    # TODO zero_grad, forward, loss, backward, step
    if epoch % 15 == 0:
        print(f"Epoch {epoch}, Loss: ?")

plt.figure(figsize=(6.4, 3.6))
plt.plot(losses)
plt.xlabel("epoch"); plt.ylabel("cross-entropy")
plt.title("training loss — scaled roster")
plt.grid(True, alpha=0.3)
plt.show()


## 7. Evaluate on the held-out 36 students

`model.eval()` + `torch.no_grad()` + `argmax(dim=1)`.

Typical result on this split after scaling: **about 94–97%** (one on-track / honors swap is common). Without scaling the same net stalls near **70%**.


In [ ]:
# TODO model.eval()
# TODO with torch.no_grad():
#         y_test_pred = model(X_test_t)
#         y_test_pred_labels = torch.argmax(y_test_pred, dim=1)

accuracy = None
print(f"Test accuracy: {accuracy}")


## 8. Alternate code — same roster, different spelling

1. **`nn.Sequential`** instead of `OutcomeNet`.
2. **SGD** instead of Adam (more epochs / slightly larger `lr`).
3. **Class-mean baseline** on the *scaled* train features — no torch.

Also train one *unscaled* Adam net so you can quote the unit-mismatch cost.


In [ ]:
# Sequential + SGD on the already-scaled tensors
seq = None
# TODO 250-epoch SGD lr=0.05, print acc

# class-mean baseline on X_train_s / X_test_s
# TODO

# unscaled Adam (raw X_train / X_test) — expect a clear drop
# TODO


## 9. More practice

**A. Support vs the rest (binary early-alert sketch).** `y_bin = (y == 0).astype(np.int64)`. Same split + scale, a `4 → 8 → 2` net, 80 epochs. Support is the separated cluster — you should land near 100%. Still not an early-alert product.

**B. XOR blobs.** `from PTEdu import xor_data`. Fit `2 → 8 → 2` for 300 epochs, Adam `lr=0.05`, then a linear `2 → 2`. Hidden net should climb; linear should hover near 50%. This is why the ReLU exists — the roster table alone will not teach that.


In [ ]:
# A. support vs rest
y_bin = (y == 0).astype(np.int64)
Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X, y_bin, test_size=0.2, random_state=0)
# TODO scale, tensors, 4→8→2, 80 epochs, print acc

# B. XOR
from PTEdu import xor_data
Xx, yx = xor_data(n=400, seed=1, noise=0.08)
# TODO hidden vs linear


## 10. Simulation — turn four knobs

Change `HIDDEN`, `LR`, `EPOCHS`, `LABEL_NOISE`, and `SCALE`.

What you should see:

- `SCALE = False` drops test accuracy into the high-60s / mid-70s on this roster.
- `LR = 0.001` often underfits in 150 epochs even when scaled.
- `HIDDEN = 4` is already enough once features are z-scored.
- `LABEL_NOISE` up to ~0.2 barely moves the test number — support is well separated; the remaining errors are the on-track / honors overlap.

`PTEdu.train_roster` applies the scale flag for you.


In [ ]:
from PTEdu import train_roster

HIDDEN = 16
LR = 0.01
EPOCHS = 150
LABEL_NOISE = 0.0
SCALE = True
SEED = 42

result = train_roster(
    X_train, y_train, X_test, y_test,
    hidden_size=HIDDEN, lr=LR, epochs=EPOCHS,
    seed=SEED, label_noise=LABEL_NOISE, scale=SCALE,
)
print(f"hidden={HIDDEN} lr={LR} epochs={EPOCHS} noise={LABEL_NOISE} scale={SCALE}")
print(f"final loss {result.losses[-1]:.4f}  test acc {result.test_acc*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(9.2, 3.6))
axes[0].plot(result.losses, color="#1A5276")
axes[0].set_title("loss"); axes[0].set_xlabel("epoch"); axes[0].grid(True, alpha=0.3)

grid_h = [4, 8, 16, 32]
grid_lr = [0.001, 0.01, 0.05]
heat = np.zeros((len(grid_h), len(grid_lr)))
for i, h in enumerate(grid_h):
    for j, lr in enumerate(grid_lr):
        heat[i, j] = train_roster(
            X_train, y_train, X_test, y_test,
            hidden_size=h, lr=lr, epochs=EPOCHS, seed=0, scale=SCALE,
        ).test_acc
im = axes[1].imshow(heat, cmap="YlGn", vmin=0.5, vmax=1.0)
axes[1].set_xticks(range(len(grid_lr)), [str(v) for v in grid_lr])
axes[1].set_yticks(range(len(grid_h)), [str(v) for v in grid_h])
axes[1].set_xlabel("lr"); axes[1].set_ylabel("hidden")
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        axes[1].text(j, i, f"{heat[i,j]:.0%}", ha="center", va="center", fontsize=8)
axes[1].set_title(f"acc heatmap @ {EPOCHS} ep, scale={SCALE}")
fig.colorbar(im, ax=axes[1], fraction=0.046)
fig.tight_layout()
plt.show()


## 11. What this model can and cannot do

**Can**
- Teach the same tensor + training-loop contract as `PTIntro`, on a roster.
- Show why education tables need train-only scaling (units do not match).
- Separate a well-spaced *support* cluster from two overlapping higher bands.
- Show that a hidden ReLU is for XOR-like structure, not for this particular roster.

**Cannot / should not**
- Assign grades, placements, scholarship cuts, or disciplinary flags.
- Stand in for an early-alert system. There is no fairness audit, no calibration plot, no counselor-in-the-loop.
- Travel to a new campus without a new split, new scale stats, and a new review process.
- Eat the Iris lab’s lunch — if every column is already in the same unit, go back to `PTIntro`.

**Four audiences (see `PTEdu_Project_Memo.docx`)**
- Learning-analytics researcher: scale + overlap, not the 97% headline.
- SIS / IR technician: dtypes, train-only scaler, eval pair.
- Dean / academic affairs: prototype on 36 hold-outs, not a product claim.
- Family / advisor: four numbers, three named bands, one likely mix-up between on-track and honors.
